# 🧪 Тестирование API

Автор: yarchegit  
Дата: May 2026  

## Описание
Полное тестирование REST API для предсказания дефолта

## ⚠️ Запуск API

Перед запуском тестов необходимо запустить API:

```bash
cd /Users/yaroslavbaev/Desktop/miphi/credit-default-service
python app/api.py
```

Или в отдельной ячейке (ниже)

In [1]:
import os
import sys
import threading
import time

os.chdir('/Users/yaroslavbaev/Desktop/miphi/credit-default-service')

def run_api():
    os.environ['AB_TEST_RATIO'] = '0.5'
    from app.api import app
    app.run(host='0.0.0.0', port=5001, debug=False, use_reloader=False)

api_thread = threading.Thread(target=run_api, daemon=True)
api_thread.start()
time.sleep(3)

print("✅ API запущен на http://localhost:5001")

[2026-05-04 23:06:30,202] INFO in api: Models loaded
{"asctime": "2026-05-04 23:06:30,202", "levelname": "INFO", "message": "Models loaded"}


 * Serving Flask app 'app.api'
 * Debug mode: off


Address already in use
Port 5001 is in use by another program. Either identify and stop that program, or start the server with a different port.


✅ API запущен на http://localhost:5001


## 1. Health Check

In [2]:
import requests
import json

r = requests.get('http://localhost:5001/health')
print(json.dumps(r.json(), indent=2))

{
  "ab_test_ratio": 0.5,
  "models": {
    "v1_loaded": true,
    "v2_loaded": true
  },
  "status": "healthy",
  "timestamp": "2026-05-04T23:06:43.664871"
}


## 2. Тестовое предсказание

In [3]:
test_data = {
    "features": {
        "LIMIT_BAL": 20000, "SEX": 2, "EDUCATION": 2, "MARRIAGE": 1,
        "AGE": 24, "PAY_0": 2, "PAY_2": 2, "PAY_3": -1, "PAY_4": -1,
        "PAY_5": -2, "PAY_6": -2, "BILL_AMT1": 3913, "BILL_AMT2": 3102,
        "BILL_AMT3": 689, "BILL_AMT4": 0, "BILL_AMT5": 0, "BILL_AMT6": 0,
        "PAY_AMT1": 0, "PAY_AMT2": 689, "PAY_AMT3": 0, "PAY_AMT4": 0,
        "PAY_AMT5": 0, "PAY_AMT6": 0
    }
}

r = requests.post('http://localhost:5001/predict', json=test_data)
print(json.dumps(r.json(), indent=2))

{
  "model_version": "v1",
  "prediction": 1,
  "prediction_label": "DEFAULT",
  "probability_default": 0.7733813279840773,
  "probability_no_default": 0.22661867201592267,
  "request_id": "20260504230651-1447",
  "risk_level": "HIGH",
  "timestamp": "2026-05-04T23:06:51.757069"
}


## 3. A/B тест - 20 запросов

In [4]:
import time

v1_count = 0
v2_count = 0

for i in range(20):
    r = requests.post('http://localhost:5001/predict', json=test_data)
    result = r.json()
    
    if result['model_version'] == 'v1':
        v1_count += 1
    else:
        v2_count += 1
    
    time.sleep(0.05)

print(f"A/B распределение (20 запросов):")
print(f"  Model v1: {v1_count} ({v1_count*5}%)")
print(f"  Model v2: {v2_count} ({v2_count*5}%)")

A/B распределение (20 запросов):
  Model v1: 11 (55%)
  Model v2: 9 (45%)


## 4. Сравнение моделей

In [5]:
r1 = requests.post('http://localhost:5001/predict/v1', json=test_data)
r2 = requests.post('http://localhost:5001/predict/v2', json=test_data)

res1 = r1.json()
res2 = r2.json()

print("Model v1:")
print(f"  Prediction: {res1['prediction_label']}")
print(f"  Probability: {res1['probability_default']:.4f}")
print(f"  Risk: {res1['risk_level']}")

print("\nModel v2:")
print(f"  Prediction: {res2['prediction_label']}")
print(f"  Probability: {res2['probability_default']:.4f}")
print(f"  Risk: {res2['risk_level']}")

diff = abs(res1['probability_default'] - res2['probability_default'])
print(f"\nРазница вероятностей: {diff:.4f}")

Model v1:
  Prediction: DEFAULT
  Probability: 0.7734
  Risk: HIGH

Model v2:
  Prediction: DEFAULT
  Probability: 0.7952
  Risk: HIGH

Разница вероятностей: 0.0218


## 5. Разные сценарии клиентов

In [6]:
# Низкий риск
low_risk = {
    "features": {
        "LIMIT_BAL": 100000, "SEX": 1, "EDUCATION": 1, "MARRIAGE": 2,
        "AGE": 35, "PAY_0": 0, "PAY_2": 0, "PAY_3": 0, "PAY_4": 0,
        "PAY_5": 0, "PAY_6": 0, "BILL_AMT1": 10000, "BILL_AMT2": 9000,
        "BILL_AMT3": 8000, "BILL_AMT4": 7000, "BILL_AMT5": 6000, "BILL_AMT6": 5000,
        "PAY_AMT1": 2000, "PAY_AMT2": 2000, "PAY_AMT3": 2000, "PAY_AMT4": 2000,
        "PAY_AMT5": 2000, "PAY_AMT6": 2000
    }
}

# Высокий риск
high_risk = {
    "features": {
        "LIMIT_BAL": 10000, "SEX": 2, "EDUCATION": 3, "MARRIAGE": 1,
        "AGE": 22, "PAY_0": 3, "PAY_2": 3, "PAY_3": 2, "PAY_4": 2,
        "PAY_5": 2, "PAY_6": 1, "BILL_AMT1": 15000, "BILL_AMT2": 14000,
        "BILL_AMT3": 13000, "BILL_AMT4": 12000, "BILL_AMT5": 11000, "BILL_AMT6": 10000,
        "PAY_AMT1": 0, "PAY_AMT2": 0, "PAY_AMT3": 0, "PAY_AMT4": 0,
        "PAY_AMT5": 0, "PAY_AMT6": 0
    }
}

r1 = requests.post('http://localhost:5001/predict', json=low_risk)
r2 = requests.post('http://localhost:5001/predict', json=high_risk)

print("Клиент с низким риском:")
print(f"  {r1.json()['prediction_label']} (prob: {r1.json()['probability_default']:.4f})")

print("\nКлиент с высоким риском:")
print(f"  {r2.json()['prediction_label']} (prob: {r2.json()['probability_default']:.4f})")

Клиент с низким риском:
  NO_DEFAULT (prob: 0.0829)

Клиент с высоким риском:
  DEFAULT (prob: 0.6545)
